# Inteligência Territorial · Manaus — painel no Colab

Roda a plataforma numa sessão **temporária** do Colab (demonstração). A sessão desliga por inatividade (até ~12 h no plano gratuito)
e o link do túnel **muda a cada execução**. Para uso contínuo/comercial, hospede em servidor próprio.

**Como usar:** menu *Ambiente de execução → Executar tudo*. No fim, a última célula imprime o link do painel.

> ⚠️ O link do túnel é público e sem login: quem tiver a URL acessa o painel (dados abertos do IBGE/OSM). A importação de imóveis fica desativada (sem `ADMIN_TOKEN`).

In [ ]:
# 1) Código e dependências (os dados derivados já vêm no repositório: não precisa refazer as ingestões)
import os
if not os.path.exists("inteligencia-territorial-manaus"):
    !git clone -q https://github.com/dsalcodaa/inteligencia-territorial-manaus
%cd inteligencia-territorial-manaus
!pip -q install -r requirements.txt

In [ ]:
# 2) Testes (opcional, ~15 s) — confirma que o ambiente do Colab reproduz os resultados
!python -m pytest -q 2>&1 | tail -3

In [ ]:
# 3) Sobe o servidor (FastAPI) em segundo plano
import subprocess, time, urllib.request
srv = subprocess.Popen(["python", "-m", "uvicorn", "app.main:app", "--host", "127.0.0.1", "--port", "8000"],
                       stdout=open("uvicorn.log", "w"), stderr=subprocess.STDOUT)
for _ in range(40):
    try:
        urllib.request.urlopen("http://127.0.0.1:8000/api/fontes", timeout=2); break
    except Exception:
        time.sleep(1)
print("servidor no ar" if srv.poll() is None else open("uvicorn.log").read())

In [ ]:
# 4) Túnel público (cloudflared, sem conta) e link do painel
import os, re, subprocess, time, urllib.request
if not os.path.exists("cloudflared"):
    urllib.request.urlretrieve("https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64", "cloudflared")
    os.chmod("cloudflared", 0o755)
tun = subprocess.Popen(["./cloudflared", "tunnel", "--url", "http://127.0.0.1:8000", "--no-autoupdate"],
                       stdout=subprocess.DEVNULL, stderr=open("cloudflared.log", "w"))
url, t0 = None, time.time()
while not url and time.time() - t0 < 90:
    time.sleep(2)
    m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", open("cloudflared.log").read())
    url = m.group(0) if m else None
print("PAINEL:", url or "túnel não subiu — veja cloudflared.log")

## Alternativa sem túnel: gerar um PDF direto no Colab
Chame o mesmo código do servidor. O arquivo é baixado pelo navegador.

In [ ]:
from google.colab import files
from app.report import build_pdf
from app.ponto import circulo, bairro_do_ponto
lon, lat, raio = -60.0217, -3.1304, 1000          # edite: ponto do empreendimento e raio (m)
emp = {"lon": lon, "lat": lat, "raio_m": raio, "nome": "Meu empreendimento", "bairro": bairro_do_ponto(lon, lat)}
open("estudo.pdf", "wb").write(build_pdf(circulo(lon, lat, raio), categoria="farmacia", empreendimento=emp))
files.download("estudo.pdf")

## Encerrar
Execute a célula abaixo (ou *Ambiente de execução → Desconectar*) para derrubar o túnel e o servidor.

In [ ]:
tun.terminate(); srv.terminate(); print("encerrado")